---
## 1. Introduction

In real-world datasets, it is extremely common to encounter **missing values** — entries that are absent, not recorded, or corrupted during data collection. These gaps can significantly impact the quality of machine learning models if not handled properly.

Missing values can arise from many sources:
- Survey respondents skipping questions
- Sensor failures or data transmission errors
- Manual data entry mistakes
- Merging datasets with different schemas

Most machine learning algorithms **cannot handle missing values natively** — they expect complete, well-formed input matrices. Therefore, handling missing data is a critical step in any preprocessing pipeline.

This notebook documents and demonstrates the `MissingValueHandler` class from the `ifri_mini_ml_lib` library using the famous **Titanic dataset**, which contains real missing values. We also compare its behavior with scikit-learn's reference implementations.

---
## 2. Clarification of Concepts

### 2.1 What is a Missing Value?

A **missing value** (also called `NaN` — *Not a Number* in Python/NumPy) is a placeholder indicating that no valid data value was stored for a variable in an observation.

In pandas and NumPy, missing values are represented as `NaN`, `None`, or `NaT` (for datetime).

### 2.2 Types of Missingness

Understanding *why* data is missing is crucial to choosing the right handling strategy. There are three main types:

| Type | Full Name | Description | Titanic Example |
|------|-----------|-------------|------------------|
| **MCAR** | Missing Completely At Random | Missingness is unrelated to any data | A few `Embarked` values missing randomly |
| **MAR** | Missing At Random | Missingness depends on other observed variables | `Age` missing more often in lower classes |
| **MNAR** | Missing Not At Random | Missingness depends on the missing value itself | `Cabin` missing mostly for 3rd class passengers |

### 2.3 Key Concepts

| Concept | Definition |
|---------|------------|
| **Imputation** | The process of replacing missing values with estimated ones |
| **Listwise deletion** | Removing entire rows that contain any missing value |
| **Mean/Median/Mode imputation** | Replacing NaN with the column's statistical summary |
| **KNN imputation** | Using the k most similar rows to estimate the missing value |
| **Regression imputation** | Predicting the missing value using a regression model |


### 2.4 Visual Overview of Strategies

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(12, 5))
ax.set_xlim(0, 10)
ax.set_ylim(0, 6)
ax.axis('off')
ax.set_facecolor('#f8f9fa')
fig.patch.set_facecolor('#f8f9fa')

ax.text(5, 5.5, 'Missing Value Handling Strategies', ha='center', va='center',
        fontsize=14, fontweight='bold', color='#2c3e50')

strategies = [
    (1.0, 3.5, '#e74c3c', 'Deletion',      'Remove rows/\ncolumns with NaN'),
    (3.0, 3.5, '#3498db', 'Statistical',   'Fill with mean,\nmedian or mode'),
    (5.0, 3.5, '#2ecc71', 'Default Value', 'Fill with a\nconstant (e.g. 0)'),
    (7.0, 3.5, '#9b59b6', 'KNN',           'Use k nearest\nneighbors'),
    (9.0, 3.5, '#e67e22', 'Regression',    'Predict with\na linear model'),
]

for x, y, color, title, desc in strategies:
    circle = plt.Circle((x, y), 0.7, color=color, alpha=0.85)
    ax.add_patch(circle)
    ax.text(x, y, title[0], ha='center', va='center', fontsize=16, fontweight='bold', color='white')
    ax.text(x, y - 1.2, title, ha='center', va='center', fontsize=9, fontweight='bold', color=color)
    ax.text(x, y - 1.85, desc, ha='center', va='center', fontsize=7.5, color='#555')

plt.title('Overview of MissingValueHandler strategies', pad=10, fontsize=11, color='#555')
plt.tight_layout()
plt.show()

---
## 3. Presentation of Algorithms

The `MissingValueHandler` class implements the following algorithms:

### 3.1 Deletion (`remove_missing`)
Removes rows or columns where the proportion of missing values exceeds a given `threshold`.
- **When to use:** When data is MCAR and the amount of missing data is small.
- **Risk:** Loss of information if too many rows/columns are dropped.

### 3.2 Statistical Imputation (`impute_statistical`)
Replaces NaN with the **mean**, **median**, or **mode** of the column.
- **Mean:** Best for normally distributed numerical data.
- **Median:** More robust to outliers.
- **Mode:** Suitable for categorical or discrete data.

### 3.3 Default Value Imputation (`impute_default`)
Replaces all NaN with a fixed constant (default: `0`).
- **When to use:** When absence of a value has a meaningful interpretation.

### 3.4 KNN Imputation (`impute_knn`)
Uses the **k-nearest neighbors** algorithm to estimate missing values based on the most similar complete rows.
- **Steps:**
  1. Separate rows with and without missing values in the target column.
  2. Train a KNN model on complete rows.
  3. Predict missing values using the incomplete rows.

### 3.5 Regression Imputation (`impute_regression`)
Fits a **Linear Regression** model on observed data to predict missing values in a target column.
- **Steps:**
  1. Use rows where the target column is not null as training data.
  2. Fit a `LinearRegression` model.
  3. Predict missing values using other features as predictors.

---
## 4. Implementation and Results

### 4.1 Setup and Loading the Titanic Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Import MissingValueHandler from the local library
from ifri_mini_ml_lib.preprocessing.preparation.missing_value_handler import MissingValueHandler

handler = MissingValueHandler()
np.random.seed(42)

# Load the Titanic dataset via seaborn (no external file needed)
import seaborn as sns
titanic_raw = sns.load_dataset('titanic')

# Select relevant columns
df = titanic_raw[['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare', 'embarked']].copy()
df.columns = ['Survived', 'Class', 'Age', 'Siblings', 'Parents', 'Fare', 'Embarked']

print("=== Titanic Dataset Overview ===")
print(df.head(10))
print(f"\nDimensions: {df.shape[0]} passengers, {df.shape[1]} columns")

### 4.2 Missing Value Analysis

In [ ]:
print("=== Missing Values per Column ===")
missing_counts = df.isnull().sum()
missing_pct    = (df.isnull().mean() * 100).round(2)
summary = pd.DataFrame({'Missing Count': missing_counts, 'Percentage (%)': missing_pct})
print(summary)
print(f"\nTotal missing: {df.isnull().sum().sum()} values out of {df.size}")

### 4.3 Visualization of Missing Data

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Missing value heatmap (first 100 rows for readability)
ax1 = axes[0]
sample = df.head(100)
im = ax1.imshow(sample.isnull().values.astype(int), cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=1)
ax1.set_xticks(range(len(df.columns)))
ax1.set_xticklabels(df.columns, rotation=30, ha='right', fontsize=9)
ax1.set_title('Missing Value Map\n(first 100 rows — red = missing)', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax1)

# Bar chart of missing counts
ax2 = axes[1]
colors = ['#e74c3c' if v > 0 else '#2ecc71' for v in missing_counts]
bars = ax2.bar(missing_counts.index, missing_counts.values, color=colors, edgecolor='white', linewidth=1.5)
ax2.set_title('Number of Missing Values per Column\n(full Titanic dataset)', fontsize=11, fontweight='bold')
ax2.set_ylabel('Count')
ax2.set_xticklabels(missing_counts.index, rotation=30, ha='right')
for bar, val in zip(bars, missing_counts.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             str(val), ha='center', va='bottom', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.show()

### 4.4 Method 1 — Deletion

In [ ]:
df_removed = handler.remove_missing(df, threshold=0.8, axis=0)

print(f"Original shape  : {df.shape}")
print(f"After deletion  : {df_removed.shape}")
print(f"Rows removed    : {df.shape[0] - df_removed.shape[0]}")
print(f"\nRemaining missing values: {df_removed.isnull().sum().sum()}")

### 4.5 Method 2 — Statistical Imputation
We work on numerical columns: `Age`, `Fare`, `Siblings`, `Parents`.

In [ ]:
df_num = df[['Age', 'Fare', 'Siblings', 'Parents']].copy()

df_mean   = handler.impute_statistical(df_num, strategy='mean')
df_median = handler.impute_statistical(df_num, strategy='median')
df_mode   = handler.impute_statistical(df_num, strategy='mode')

missing_age_idx = df_num['Age'].isnull()

print("Imputed values for 'Age' (rows that were missing):")
comparison = pd.DataFrame({
    'Original': df_num['Age'],
    'Mean':     df_mean['Age'],
    'Median':   df_median['Age'],
    'Mode':     df_mode['Age'],
})
print(comparison[missing_age_idx].head(10))

In [ ]:
# Distribution of Age before and after imputation
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
methods = [('Mean', df_mean, '#3498db'), ('Median', df_median, '#e74c3c'), ('Mode', df_mode, '#2ecc71')]

for ax, (name, df_imp, color) in zip(axes, methods):
    ax.hist(df_num['Age'].dropna(), bins=20, alpha=0.5, label='Original', color='#95a5a6', edgecolor='white')
    ax.hist(df_imp['Age'],          bins=20, alpha=0.6, label=f'Imputed ({name})', color=color, edgecolor='white')
    ax.set_title(f'Age Distribution\nafter {name} imputation', fontweight='bold')
    ax.set_xlabel('Age')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Impact of Statistical Imputation on Age Distribution — Titanic',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 4.6 Method 3 — Default Value

In [ ]:
df_default = handler.impute_default(df_num, value=-1)

print("Default value imputation (-1) — rows that were missing in 'Age':")
print(df_default[missing_age_idx][['Age']].head(10))

### 4.7 Method 4 — KNN Imputation

In [ ]:
# Use a subset to limit computation time
df_knn_input = df_num.head(100).copy()
df_knn = handler.impute_knn(df_knn_input, k=5, task='regression')

print("KNN Imputation (k=5) — columns Age, Fare, Siblings, Parents:")
print(f"Missing values before: {df_knn_input.isnull().sum().sum()}")
print(f"Missing values after : {df_knn.isnull().sum().sum()}")
print("\nSample imputed values for 'Age':")
missing_knn = df_knn_input['Age'].isnull()
print(df_knn[missing_knn][['Age']].head())

### 4.8 Method 5 — Regression Imputation

In [ ]:
# Predict Age based on Class, Fare, Siblings, Parents
df_reg_input = df[['Class', 'Age', 'Siblings', 'Parents', 'Fare']].copy()
df_reg = handler.impute_regression(df_reg_input, target_col='Age')

print("Regression Imputation (target = 'Age'):")
print(f"Missing values before: {df_reg_input['Age'].isnull().sum()}")
print(f"Missing values after : {df_reg['Age'].isnull().sum()}")
print("\nSample predicted values:")
missing_reg = df_reg_input['Age'].isnull()
print(df_reg[missing_reg][['Age']].head(10))

### 4.9 Visual Comparison of All Methods on 'Age'

In [ ]:
df_reg_age = handler.impute_regression(
    df[['Class','Age','Siblings','Parents','Fare']].copy(), target_col='Age'
)['Age']

fig, ax = plt.subplots(figsize=(12, 5))

ax.hist(df['Age'].dropna(), bins=30, alpha=0.4,  label='Original (no NaN)', color='#2c3e50', edgecolor='white')
ax.hist(df_mean['Age'],     bins=30, alpha=0.5,  label='Mean',              color='#3498db', edgecolor='white')
ax.hist(df_median['Age'],   bins=30, alpha=0.5,  label='Median',            color='#e74c3c', edgecolor='white')
ax.hist(df_reg_age,         bins=30, alpha=0.5,  label='Regression',        color='#e67e22', edgecolor='white')

ax.set_title('Age Distribution by Imputation Method — Titanic', fontsize=12, fontweight='bold')
ax.set_xlabel('Age')
ax.set_ylabel('Frequency')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 5. Scikit-learn Performance Visualization

In [ ]:
from sklearn.impute import SimpleImputer, KNNImputer

X = df_num.values

sk_mean   = SimpleImputer(strategy='mean').fit_transform(X)
sk_median = SimpleImputer(strategy='median').fit_transform(X)
sk_mode   = SimpleImputer(strategy='most_frequent').fit_transform(X)
sk_knn    = KNNImputer(n_neighbors=5).fit_transform(X)

print("scikit-learn SimpleImputer (mean) — first 5 rows:")
print(pd.DataFrame(sk_mean, columns=['Age','Fare','Siblings','Parents']).head().round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# sklearn
ax1 = axes[0]
ax1.hist(df['Age'].dropna(), bins=30, alpha=0.4,  label='Original',    color='#2c3e50', edgecolor='white')
ax1.hist(sk_mean[:,0],       bins=30, alpha=0.55, label='sklearn Mean', color='#3498db', edgecolor='white')
ax1.hist(sk_median[:,0],     bins=30, alpha=0.55, label='sklearn Median',color='#e74c3c', edgecolor='white')
ax1.hist(sk_knn[:,0],        bins=30, alpha=0.55, label='sklearn KNN',  color='#9b59b6', edgecolor='white')
ax1.set_title('Scikit-learn — Imputed Age Distribution', fontweight='bold')
ax1.set_xlabel('Age')
ax1.set_ylabel('Frequency')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# IFRI
ax2 = axes[1]
ax2.hist(df['Age'].dropna(), bins=30, alpha=0.4,  label='Original',   color='#2c3e50', edgecolor='white')
ax2.hist(df_mean['Age'],     bins=30, alpha=0.55, label='IFRI Mean',   color='#3498db', edgecolor='white')
ax2.hist(df_median['Age'],   bins=30, alpha=0.55, label='IFRI Median', color='#e74c3c', edgecolor='white')
ax2.set_title('IFRI Mini ML Lib — Imputed Age Distribution', fontweight='bold')
ax2.set_xlabel('Age')
ax2.set_ylabel('Frequency')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.suptitle('Distribution Comparison — Titanic (Age column)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6. Differences between `ifri_mini_ml_lib` and `scikit-learn`

### 6.1 Execution Time Bar Chart

In [1]:
import time

def benchmark(fn, *args, n=10):
    """Run fn n times and return average execution time in ms."""
    times = []
    for _ in range(n):
        t0 = time.perf_counter()
        fn(*args)
        times.append((time.perf_counter() - t0) * 1000)
    return np.mean(times)

X_bench = df_num.values

t_ifri_mean = benchmark(handler.impute_statistical, df_num.copy(), 'mean')
t_ifri_med  = benchmark(handler.impute_statistical, df_num.copy(), 'median')
t_ifri_mode = benchmark(handler.impute_statistical, df_num.copy(), 'mode')
t_ifri_def  = benchmark(handler.impute_default,     df_num.copy(), 0)

t_sk_mean = benchmark(SimpleImputer(strategy='mean').fit_transform,          X_bench)
t_sk_med  = benchmark(SimpleImputer(strategy='median').fit_transform,        X_bench)
t_sk_mode = benchmark(SimpleImputer(strategy='most_frequent').fit_transform, X_bench)
t_sk_def  = benchmark(SimpleImputer(strategy='constant', fill_value=0).fit_transform, X_bench)

labels = ['Mean', 'Median', 'Mode', 'Constant']
t_ifri = [t_ifri_mean, t_ifri_med, t_ifri_mode, t_ifri_def]
t_sk   = [t_sk_mean,   t_sk_med,   t_sk_mode,   t_sk_def]

x_pos = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x_pos - width/2, t_ifri, width, label='ifri_mini_ml_lib', color='#3498db', edgecolor='white')
b2 = ax.bar(x_pos + width/2, t_sk,   width, label='scikit-learn',     color='#e74c3c', edgecolor='white')

for b in b1:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.05,
            f'{b.get_height():.2f}', ha='center', va='bottom', fontsize=8, color='#3498db')
for b in b2:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.05,
            f'{b.get_height():.2f}', ha='center', va='bottom', fontsize=8, color='#e74c3c')

ax.set_title('Execution Time Comparison (ms) — Titanic dataset (891 rows)', fontsize=11, fontweight='bold')
ax.set_xlabel('Imputation Strategy')
ax.set_ylabel('Average Time (ms)')
ax.set_xticks(x_pos)
ax.set_xticklabels(labels)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

NameError: name 'df_num' is not defined

### 6.2 Feature Comparison Table

In [ ]:
comparison = pd.DataFrame({
    'Feature': [
        'Mean imputation',
        'Median imputation',
        'Mode imputation',
        'Constant imputation',
        'KNN imputation',
        'Regression imputation',
        'Works on DataFrames natively',
        'Works on numpy arrays',
        'Handles mixed types',
        'Fit/Transform API',
        'Pipeline compatible',
        'Optimized for large datasets'
    ],
    'ifri_mini_ml_lib': [
        'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes',
        'Yes', 'Yes (converted)', 'Partial', 'No', 'No', 'No'
    ],
    'scikit-learn': [
        'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No (IterativeImputer only)',
        'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes'
    ]
})

print(comparison.to_string(index=False))

---
## 7. Conclusion

Using the **Titanic dataset**, we demonstrated the five missing value handling strategies implemented in `MissingValueHandler`:

- **Deletion** efficiently reduces incomplete data but at the cost of information loss.
- **Statistical imputation** (mean/median/mode) is simple, fast, and produces results **numerically identical** to scikit-learn.
- **Default value imputation** is useful when absence carries semantic meaning.
- **KNN imputation** provides more accurate estimates by leveraging inter-feature relationships.
- **Regression imputation** is unique to IFRI among simple methods — scikit-learn only offers this via `IterativeImputer` (experimental).

In terms of performance, scikit-learn remains **faster** on large datasets thanks to its internal C/Cython optimizations. However, `ifri_mini_ml_lib` is an excellent **educational tool** for deeply understanding imputation mechanisms, with clear, readable, and extensible code.

---
## 8. Sources

- Pedregosa et al. (2011). *Scikit-learn: Machine Learning in Python*. JMLR 12, pp. 2825–2830. https://scikit-learn.org
- McKinney, W. (2010). *Data Structures for Statistical Computing in Python*. Proceedings of the 9th Python in Science Conference.
- Little, R.J.A. & Rubin, D.B. (2002). *Statistical Analysis with Missing Data* (2nd ed.). Wiley.
- Waskom, M. (2021). *Seaborn: statistical data visualization*. Journal of Open Source Software. https://seaborn.pydata.org
- scikit-learn documentation — `SimpleImputer`: https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html
- scikit-learn documentation — `KNNImputer`: https://scikit-learn.org/stable/modules/generated/sklearn.impute.KNNImputer.html
- Titanic Dataset: https://www.kaggle.com/c/titanic
- IFRI Mini ML Lib GitHub: https://github.com/IFRI-AI-Classes/ifri_mini_ml_lib
